# PPO с Продвинутыми Архитектурами (GRU / CNN)

В этом ноутбуке мы используем **Frame Stacking** (история из $N$ свечей) и подаем ее в рекуррентные (GRU) или сверточные (CNN) сети. Механизм Gating (FiLM) от HMM остается активным.

In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext tensorboard

import os
import wandb
import pandas as pd
import numpy as np
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack
from stable_baselines3.common.monitor import Monitor
from wandb.integration.sb3 import WandbCallback

from core.data.data_loader import load_crypto_data
from core.features.feature_generator import FeatureGenerator
from custom_envs.trading_env_v5 import TradingEnvV5
from agents.ppo_agent import create_ppo_agent, TradingMetricsCallback

# Инициализация Weights & Biases
run = wandb.init(
    project="trading_rl",
    name="ppo_gated_gru", # Измените на cnn, если учите свертки
    sync_tensorboard=True,
    monitor_gym=True,
    save_code=True,
)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

print("Настройка путей для сохранения/загрузки моделей:")
model_widget = widgets.Text(value="models/ppo_gated_agent_final", description="Model Path:", layout=widgets.Layout(width='50%'))
norm_widget = widgets.Text(value="models/vec_normalize_final.pkl", description="Norm Path:", layout=widgets.Layout(width='50%'))
display(model_widget, norm_widget)

## 1. Загрузка данных и генерация фичей (Включая HMM)

In [ ]:
df = load_crypto_data(symbol='BTCUSDT', start_date='2024-01-01', interval='4h', source='bybit_futures')

fg = FeatureGenerator()
data_features = fg.transform(df)

hmm_cols = [c for c in data_features.columns if 'hmm_regime' in c]
print(f"Data Features shape: {data_features.shape}")

## 2. Создание среды со Стеком Кадров (Frame Stacking)
`VecFrameStack` автоматически дублирует первый кадр в начале эпизода, поэтому нам не нужно вручную делать прогрев среды. Вектор наблюдения увеличится ровно в `n_stack` раз.

In [ ]:
def make_env():
    env = TradingEnvV5(df=data_features, continuous_action=True, t_max=1000)
    return Monitor(env)

vec_env = DummyVecEnv([make_env])

# ОЧЕНЬ ВАЖНО: Добавляем Frame Stacking ДО нормализации
N_STACK = 10
vec_env = VecFrameStack(vec_env, n_stack=N_STACK)

# Нормализация
vec_env = VecNormalize(
    vec_env,
    norm_obs=True,
    norm_reward=True,
    clip_obs=10.,
    clip_reward=10.
)

## 3. Запуск TensorBoard

In [ ]:
%tensorboard --logdir ./tensorboard_logs/

## 4. Создание Агента (GRU или CNN)

In [ ]:
# Вы можете поменять 'gru' на 'cnn'
model = create_ppo_agent(
    vec_env, 
    extractor_type="gru", 
    num_hmm_states=len(hmm_cols), 
    n_stack=N_STACK,
    tensorboard_log="./tensorboard_logs/"
)

## 5. Обучение

In [ ]:
callbacks = [
    TradingMetricsCallback(),
    WandbCallback(
        gradient_save_freq=1000,
        model_save_path=f"models/{run.id}",
        verbose=0,
    )
]

print("Начинаем обучение PPO с продвинутой архитектурой...")
model.learn(total_timesteps=500_000, callback=callbacks, progress_bar=True)

model.save(model_widget.value)
vec_env.save(norm_widget.value)
print("Модель сохранена!")
wandb.finish()

## 🔬 Универсальный Блок Валидации
Этот блок можно запускать отдельно в любой момент. Он загружает обученную модель с диска, прогоняет её по тестовому куску данных и рисует красивый график цены, совмещенный с кривой вашего депозита.


In [ ]:

import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack
from custom_envs.trading_env_v5 import TradingEnvV5
from core.visualization.visualizer_candles import CandleChartVisualizer

def get_base_env(vec_env):
    while hasattr(vec_env, 'venv'):
        vec_env = vec_env.venv
    return vec_env.envs[0]

# ==========================================
# НАСТРОЙКИ МОДЕЛИ И ДАННЫХ
# ==========================================
MODEL_PATH = "models/ppo_gated_agent_final"         # <--- Укажите путь к модели
NORM_PATH = "models/vec_normalize_final.pkl"  # <--- Укажите путь к нормализатору

# ВАЖНО: Если загружаете модель из 7 ноутбука (MLP), ставьте False. Для 8 и 10 (GRU/CNN) - True.
USE_FRAME_STACK = True
N_STACK = 10
TEST_SIZE = 360 # Последние 360 свечей (2 месяца)
# ==========================================

try:
    model = PPO.load(MODEL_PATH)
    print(f"✅ Модель {MODEL_PATH} успешно загружена.")
except Exception as e:
    print(f"⚠️ Ошибка загрузки модели: {e}. Проверьте пути!")

# Берем последние TEST_SIZE свечей для теста
test_df = data_features.iloc[-TEST_SIZE:].reset_index(drop=True)

test_env = TradingEnvV5(df=test_df, continuous_action=True, t_max=len(test_df), initial_deposit=100_000.0)
vec_env = DummyVecEnv([lambda: test_env])

if USE_FRAME_STACK:
    vec_env = VecFrameStack(vec_env, n_stack=N_STACK)

try:
    vec_env = VecNormalize.load(NORM_PATH, vec_env)
    vec_env.training = False
    vec_env.norm_reward = False
except Exception as e:
    print(f"⚠️ Ошибка загрузки нормализатора: {e}")

obs = vec_env.reset()
done = [False]

actions_list = []
portfolio_values = []

print("⏳ Запускаем симуляцию торгов...")
while not done[0]:
    action, _ = model.predict(obs, deterministic=True)
    obs, _, done, info = vec_env.step(action)
    
    # Расшифровываем continuous action для визуализатора
    act_val = action[0][0] if isinstance(action, np.ndarray) else action
    if act_val > 0.1:
        actions_list.append('buy')
    elif act_val < -0.1:
        actions_list.append('sell')
    else:
        actions_list.append('hold')
        
    real_env = get_base_env(vec_env)
    cur_price = real_env._get_current_price()
    portfolio_values.append(real_env.get_portfolio_value(cur_price))

# --- Отчет ---
print(f"\n{'='*30}")
print(f"📊 ОТЧЕТ О ТОРГОВЛЕ (Out-of-Sample)")
print(f"{'='*30}")
print(f"Начальный баланс: $100,000.00")
print(f"Финальный баланс: ${portfolio_values[-1]:,.2f}")
roi = (portfolio_values[-1] - 100_000) / 100_000 * 100
print(f"Чистая прибыль:   {roi:.2f}%")

active_steps = len([a for a in actions_list if a != 'hold'])
print(f"Свечей в позиции: {active_steps} из {len(actions_list)}")

if real_env.current_step >= len(test_df) - 1:
    print("Статус: 🏁 Успешно дошел до конца отрезка")
else:
    print("Статус: 💀 Ликвидирован (Margin Call / Max Drawdown)")

# --- Визуализация ---
actions_series = pd.Series(actions_list)
port_series = pd.Series(portfolio_values)

visualizer = CandleChartVisualizer(use_volume_width=False)
visualizer.plot_candlestick(
    data=test_df.iloc[:len(actions_list)], 
    title=f"Test Backtest ({roi:.2f}% PnL)", 
    actions=actions_series,
    portfolio_values=port_series
)


## 🔬 Валидация (Через Утилиту)
Этот блок использует нашу универсальную функцию валидации. При каждом запуске он выбирает случайный кусок графика (если случайный старт включен) и показывает результат торговли.


In [ ]:

from core.evaluation.validation import run_validation

# Запускаем симуляцию случайного отрезка графика
run_validation(
    data_features=data_features,
    model_path=model_widget.value,
    norm_path=norm_widget.value,
    use_frame_stack=True,
    n_stack=10,
    test_size=360,
    random_start=True
)
